# Quickstart Guide

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chaobrain/saiunit/blob/master/docs/getting_started/quickstart.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/chaobrain/saiunit/blob/master/docs/getting_started/quickstart.ipynb)

**saiunit** is a unit-aware scientific computing library built on JAX.
It tracks physical units through all computations — arithmetic, linear algebra, FFTs,
automatic differentiation, and JIT compilation — catching dimension errors at runtime.

This guide covers the essentials in 5 minutes.

## Installation

```bash
pip install saiunit
```

In [ ]:
import saiunit as u
import jax
import jax.numpy as jnp

## Creating Quantities

A `Quantity` = numeric value + physical unit. Create one by multiplying a value with a unit.

In [ ]:
# Scalars
mass = 5.0 * u.kilogram
speed = 10.0 * u.meter / u.second
print('mass:', mass)
print('speed:', speed)

In [ ]:
# Arrays
voltages = jnp.array([1.0, 2.5, 3.7]) * u.mV
print('voltages:', voltages)
print('shape:', voltages.shape, 'dtype:', voltages.dtype)

In [ ]:
# Direct construction
current = u.Quantity(jnp.array([0.1, 0.2, 0.3]), unit=u.ampere)
print('current:', current)

## Arithmetic with Units

Units are tracked automatically. Incompatible operations raise errors.

In [ ]:
# Addition: same dimension required
t1 = 500.0 * u.ms
t2 = 1.5 * u.second
print('t1 + t2:', t1 + t2)  # auto-aligns to first unit

In [ ]:
# Multiplication: units multiply
F = 10.0 * u.newton
d = 3.0 * u.meter
print('work = F * d:', F * d)  # N * m = J

In [ ]:
# Division: units divide
print('speed = d / t:', (100.0 * u.meter) / (10.0 * u.second))  # m/s

In [ ]:
# Dimension mismatch raises error
try:
    result = 5.0 * u.meter + 3.0 * u.second
except Exception as e:
    print('Error:', e)

## Unit Conversion

Use `to_decimal()` to extract the numeric value in a target unit,
or `in_unit()` to get a new Quantity in the target unit.

In [ ]:
distance = 2.5 * u.kmeter
print('In meters:', distance.to_decimal(u.meter))       # 2500.0
print('In cm:', distance.to_decimal(u.cmeter))           # 250000.0
print('As Quantity:', distance.in_unit(u.meter))          # 2500.0 m

## Quantity Attributes

In [ ]:
q = jnp.array([[1., 2.], [3., 4.]]) * u.volt
print('mantissa:', q.mantissa)   # numeric array
print('unit:', q.unit)           # the unit
print('dim:', q.dim)             # physical dimension
print('shape:', q.shape)         # array shape
print('dtype:', q.dtype)         # array dtype

## Unit-Aware Math Functions

`saiunit.math` provides 500+ functions that understand units.

In [ ]:
data = jnp.array([2., 4., 6., 8., 10.]) * u.newton

print('sum:', u.math.sum(data))           # keeps unit
print('mean:', u.math.mean(data))         # keeps unit
print('sqrt:', u.math.sqrt(4.0 * u.meter2))  # changes unit: m^2 -> m
print('sort:', u.math.sort(jnp.array([3., 1., 2.]) * u.volt))

## Physical Constants

In [ ]:
from saiunit import constants

print('Avogadro number:', constants.avogadro)
print('Boltzmann constant:', constants.boltzmann)
print('Elementary charge:', constants.elementary_charge)
print('Electron mass:', constants.electron_mass)

## JAX Transforms: `jit`, `vmap`, `grad`

Quantities work seamlessly with JAX transformations.

In [ ]:
# JIT compilation
@jax.jit
def kinetic_energy(m, v):
    return 0.5 * m * v**2

KE = kinetic_energy(2.0 * u.kilogram, 3.0 * u.meter / u.second)
print('KE =', KE)  # kg * m^2 / s^2 = J

In [ ]:
# vmap: vectorize over a batch
velocities = jnp.array([1., 2., 3., 4., 5.]) * u.meter / u.second
energies = jax.vmap(lambda v: kinetic_energy(2.0 * u.kilogram, v))(velocities)
print('Batch KE:', energies)

In [ ]:
# grad: automatic differentiation with unit tracking
dKE_dv = u.autograd.grad(lambda v: 0.5 * (2.0 * u.kilogram) * v**2)
print('dKE/dv at v=3 m/s:', dKE_dv(3.0 * u.meter / u.second))  # momentum: kg * m/s

## Unit Validation with Decorators

Use `@check_units` to enforce unit contracts on function arguments.

In [ ]:
@u.check_units(v=u.meter / u.second, t=u.second)
def displacement(v, t):
    return v * t

print('displacement:', displacement(10.0 * u.meter / u.second, 5.0 * u.second))

In [ ]:
# Wrong units raise an error
try:
    displacement(10.0 * u.kilogram, 5.0 * u.second)
except Exception as e:
    print('Error:', e)

## What's Next?

- **[Quantity](../physical_units/quantity.ipynb)** — Creating and manipulating quantities in depth
- **[Standard Units](../physical_units/standard_units.ipynb)** — All available SI and non-SI units
- **[Unit Conversion](../physical_units/conversion.ipynb)** — Converting between units
- **[NumPy Functions](../mathematical_functions/numpy_functions.ipynb)** — 500+ unit-aware math functions
- **[Linear Algebra](../mathematical_functions/linalg_functions.ipynb)** — Unit-aware linalg
- **[Unit Validation](../mathematical_functions/check_units.ipynb)** — check_dims and check_units decorators